Juan Manuel Tabares Uribe

Sara Mesa Gómez

In [ ]:
import kagglehub
import os
import pandas as pd
path = kagglehub.dataset_download("suhanigupta04/student-placement-prediction-dataset")
print("Path to dataset files:", path)
csv_file = [f for f in os.listdir(path) if f.endswith('.csv')][0]
df = pd.read_csv(os.path.join(path, csv_file))

Using Colab cache for faster access to the 'student-placement-prediction-dataset' dataset.
Path to dataset files: /kaggle/input/student-placement-prediction-dataset


# 4-5. Preparación y separación de los datos
- Transformaciones y decisiones

Prevención de Fuga de Información (Data Leakage):
Se realizó la partición del conjunto de datos en 80% entrenamiento y 20% prueba antes de aplicar cualquier transformación.
Las métricas estadísticas (como la mediana para imputar o los parámetros $\mu$ y $\sigma$ para escalar) se calcularon sobre X_train y se aplicaron a X_test.

- Separación de entrenamiento y prueba

Porcentajes: 80% para entrenamiento (80.000 filas) y 20% para prueba (20.000 filas).

Método: se utilizó train_test_split de forma aleatoria y estratificada por placement_status (stratify=y), con random_state=42. La estratificación mantiene la misma proporción de contratados y no contratados (≈ 68% / 32%) en ambos conjuntos, y la semilla hace que la división sea reproducible.

Grupos naturales: cada fila corresponde a un estudiante distinto; no hay identificadores ni varios registros de la misma persona. Por eso, una separación aleatoria no reparte información de un mismo estudiante entre entrenamiento y prueba.

- Exclusión de la variable salary_package_lpa: Se excluyó la variable salary_package_lpa porque solo existe cuando el estudiante fue contratado. Sus 31.525 valores nulos coinciden exactamente con los 31.525 estudiantes no contratados, es decir, es una variable que se conoce después del evento que se quiere predecir y no estaría disponible al momento de hacer una predicción (ver sección 6).

- Selección de variables: Se conservaron las 16 predictoras restantes. Ningún par de variables numéricas supera |r| > 0,8 (sección 3), por lo que no hay redundancia que justifique eliminar alguna, y todas están disponibles antes de la decisión de contratación.

- Valores Faltantes

Origen y cuantificación: El conjunto de datos original no tiene valores faltantes en las predictoras, por lo que se simularon de forma aleatoria (completamente al azar) y reproducible (semilla 42): un 5% en cada una de las 16 predictoras. En la celda siguiente se cuantifican los nulos por variable.

Impacto: Como los nulos están repartidos al azar entre 16 columnas, cerca del 56% de las filas tiene al menos un valor faltante, y eliminarlas dejaría solo alrededor de 44.000 de las 100.000 observaciones (ver la salida de la celda siguiente).

Variables Numéricas: Se seleccionó la imputación por la mediana en lugar de la media debido a que valores atípicos no afectan la mediana. Además, en estas variables la media y la mediana casi coinciden (por ejemplo, cgpa 7,21 / 7,21 y aptitude_score 64,99 / 65,00), así que la imputación no desplaza la distribución.

Variables Categóricas: Se utilizó la imputación por la moda, con la estructura categórica original sin perder registros.

No eliminación de registros: No se eliminaron las filas con datos faltantes porque esto habría reducido el tamaño de la muestra a menos de la mitad y recortado la capacidad de generalización del modelo. Además, como los nulos son aleatorios, eliminarlos no aporta ninguna ventaja.

- Codificación de Variables Categóricas:
Se aplicó One-Hot Encoding para las características cualitativas nominales. Esto permite que los modelos basados en álgebra lineal e interpretación geométrica procesen las categorías como vectores binarios independientes sin asumir un falso orden numérico implícito. En branch las categorías no tienen orden. En college_tier sí existe un orden (Tier-1 > Tier-2 > Tier-3), pero con One-Hot no se supone que la distancia entre Tier-1 y Tier-2 sea igual a la de Tier-2 a Tier-3.

- Escalamiento de Características:
Se aplicó StandardScaler para estandarizar las variables numéricas a media 0 y varianza 1. Esto previene que variables con rangos numéricos elevados (como aptitude_score, que va de 20 a 100) dominen sobre variables de escalas pequeñas (como promedios, puntajes de 0 a 10 o conteos) durante el cálculo de coeficientes y distancias del modelo.

- Valores atípicos:
No se eliminaron. Los valores extremos observados en los diagramas de caja (por ejemplo, cgpa < 4,5 o backlogs = 3) están dentro del rango válido de cada variable y corresponden a estudiantes reales poco comunes, no a errores de registro.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

#Se excluye la variable del salario
X = df.drop(columns=['placement_status', 'salary_package_lpa']).copy()
#Separar la variable objetivo
y = df['placement_status']
print(f"Predictoras: {X.shape[1]} | Observaciones: {X.shape[0]}\n")

# Simulación de valores faltantes (5 % aleatorio en cada predictora, semilla 42)
np.random.seed(42)
mascara = np.random.rand(*X.shape) < 0.05
X = X.mask(mascara)

# Cuantificación de los nulos por variable
tabla_nulos = pd.DataFrame({
    'Cantidad_Nulos': X.isnull().sum(),
    'Porcentaje_%': (X.isnull().mean() * 100).round(2)
})
print("VALORES FALTANTES POR VARIABLE")
print(tabla_nulos)

# Impacto de eliminar las filas con nulos
filas_con_nulo = X.isnull().any(axis=1).mean() * 100
print(f"\nFilas con al menos un nulo: {filas_con_nulo:.2f} %")
print(f"Filas que quedarían si se eliminaran las incompletas: {X.dropna().shape[0]} de {X.shape[0]}")

# Efecto sobre la distribución: media y mediana originales frente a la media con nulos
cols_num = X.select_dtypes('number').columns
comparacion = pd.DataFrame({
    'Media_original': df[cols_num].mean(),
    'Mediana_original': df[cols_num].median(),
    'Media_con_nulos': X[cols_num].mean()
}).round(2)
print("\nCOMPARACIÓN DE LA DISTRIBUCIÓN")
print(comparacion)

print("\nModa de las categóricas:", {c: X[c].mode()[0] for c in X.select_dtypes('object').columns})

Predictoras: 16 | Observaciones: 100000

VALORES FALTANTES POR VARIABLE
                           Cantidad_Nulos  Porcentaje_%
branch                               4885          4.88
college_tier                         5034          5.03
cgpa                                 4942          4.94
backlogs                             4966          4.97
coding_skills                        4823          4.82
dsa_score                            4796          4.80
aptitude_score                       4909          4.91
communication_skills                 4951          4.95
ml_knowledge                         5006          5.01
system_design                        4920          4.92
internships                          5054          5.05
projects_count                       5131          5.13
certifications                       5033          5.03
hackathons                           5036          5.04
open_source_contributions            5053          5.05
extracurriculars                

In [ ]:
#Separación de datos: 80% Entrenamiento y 20% Prueba
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f"Entrenamiento: {X_train.shape[0]} filas ({X_train.shape[0]/len(X):.0%})")
print(f"Prueba: {X_test.shape[0]} filas ({X_test.shape[0]/len(X):.0%})\n")
print("Proporción de la variable objetivo en cada conjunto (%):")
print(pd.DataFrame({
    'Train_%': (y_train.value_counts(normalize=True) * 100).round(2),
    'Test_%': (y_test.value_counts(normalize=True) * 100).round(2)
}))

# Transformaciones
num_cols = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
print(f"\nVariables Numéricas ({len(num_cols)}): {num_cols}")
print(f"Variables Categóricas ({len(cat_cols)}): {cat_cols}")

# Numéricas: imputación por mediana + estandarización
pipeline_numerico = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Categóricas: imputación por moda + One-Hot
pipeline_categorico = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocesador = ColumnTransformer(transformers=[
    ('num', pipeline_numerico, num_cols),
    ('cat', pipeline_categorico, cat_cols)
])

# fit con entrenamiento y transform en prueba con los parámetros aprendidos en entrenamiento
X_train_preprocesado = preprocesador.fit_transform(X_train)
X_test_preprocesado = preprocesador.transform(X_test)

nombres_cat = preprocesador.named_transformers_['cat']['encoder'].get_feature_names_out(cat_cols)
columnas_totales = num_cols + list(nombres_cat)
X_train_ready = pd.DataFrame(X_train_preprocesado, columns=columnas_totales)
X_test_ready = pd.DataFrame(X_test_preprocesado, columns=columnas_totales)

print("\nTRANSFORMACIÓN")
print(f"X_train: {X_train_ready.shape} | X_test: {X_test_ready.shape}")
print("Nulos restantes después de imputar:", int(X_train_ready.isnull().sum().sum()))

Entrenamiento: 80000 filas (80%)
Prueba: 20000 filas (20%)

Proporción de la variable objetivo en cada conjunto (%):
                  Train_%  Test_%
placement_status                 
1                   68.47   68.47
0                   31.52   31.52

Variables Numéricas (14): ['cgpa', 'backlogs', 'coding_skills', 'dsa_score', 'aptitude_score', 'communication_skills', 'ml_knowledge', 'system_design', 'internships', 'projects_count', 'certifications', 'hackathons', 'open_source_contributions', 'extracurriculars']
Variables Categóricas (2): ['branch', 'college_tier']

TRANSFORMACIÓN
X_train: (80000, 24) | X_test: (20000, 24)
Nulos restantes después de imputar: 0
